# Term Frequency–Inverse Document Frequency (TF–IDF)
## 01. Introduction

Consider the following academic abstracts:

**Paper A (Machine Learning)**
> “This paper presents a novel neural network architecture for image classification. The network uses convolutional layers…”

**Paper B (Cryptography)**
> “This paper presents a new encryption algorithm for secure communication. The algorithm uses prime factorization…”

How could a model tell these apart?

A naive approach is to simply count how many times each word appears and compare those counts:

|           | this | paper | presents | neural | network | encryption | algorithm |
|:----------|:----:|:-----:|:--------:|:------:|:-------:|:----------:|:---------:|
| **Paper A** | 1 | 1 | 1 | 2 | 2 | 0 | 0 |
| **Paper B** | 1 | 1 | 1 | 0 | 0 | 2 | 2 |

The issue is clear: words like *“this,” “paper,”* and *“presents”* appear everywhere and carry no distinctive meaning.
The words *“neural,” “network,” “encryption,”* and *“algorithm”* actually define the topics, yet all words are treated equally in a raw count.

TF–IDF (Term Frequency–Inverse Document Frequency) solves this by rewarding words that are **frequent within a document** but **rare across the corpus**.
It measures two complementary properties of words:

---

### 1.1 Term Frequency (TF)

$$
\mathrm{TF}(w, d) \;=\; \frac{\text{count}(w,d)}{\sum_{w'} \text{count}(w',d)}
$$

**Example:**
If Paper A has 100 words and *“neural”* appears twice, then
$$\mathrm{TF}(\text{“neural”}, \text{Paper A}) = \tfrac{2}{100} = 0.02.$$

---

### 1.2 Inverse Document Frequency (IDF)

$$
\mathrm{IDF}(w) \;=\; \log\!\left(\frac{N}{n_w}\right)
$$

Where \(N\) is the total number of documents and \(n_w\) is the number of documents containing word \(w\).

**Example:** Using a corpus of \(N=1000\) papers:

| Word | Docs Containing | IDF |
|:-----|:----------------:|:---:|
| the | 1000 | \(\log(1000/1000) = 0\) |
| convolutional | 50 | \(\log(1000/50) \approx 3\) |

Common words receive low IDF ≈ 0; rare words receive high IDF values.
*(Many libraries use smoothing, e.g., \(\log\!\big(\frac{N+1}{n_w+1}\big)+1\), to avoid edge cases.)*

---

### 1.3 Combining Them

$$
\mathrm{TF\text{-}IDF}(w, d) \;=\; \mathrm{TF}(w,d)\times \mathrm{IDF}(w)
$$

This produces a weighted vector where distinctive terms get higher scores and common filler terms fade toward zero.
The resulting vectors are more **separable**, allowing classifiers to distinguish topics like machine learning versus cryptography using only word statistics.

---
### 1.4 Important Topics

#### Stop Words
Stop words are widespread words used to glue sentences together but tell us little about the meaning of the text. Such words include `the`, `is`, `a`, `for`, etc., and appear in all forms of literature including fiction, research, blogging, etc. These values will score very low and therefore influence very little to the final classification. In training stop words were set to english to remove these words from the text when creating the bag of words enabling the model to only consider more influential words.

#### Sparsity

This foundation will let us explore how TF–IDF transforms raw text into meaningful numeric representations for downstream models such as logistic regression, SVMs, or neural classifiers.

---

### 1.5 Imports

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

## 02. Load & Split Data

In [2]:
print("Splitting data: using val in training data for final training")
data = pd.read_parquet('../data/processed/arxiv_text.parquet')

X_train = data[data['split'] == 'train']
y_train = data[data['split'] == 'train'].label

X_val = data[data['split'] == 'valid']
y_val = data[data['split'] == 'valid'].label

X_test = data[data['split'] == 'test']
y_test = data[data['split'] == 'test'].label

print(f"The shape of the training data is X train: {X_train.shape} and y train: {y_train.shape}")
print(f"The shape of the training data is X val: {X_val.shape} and y val: {y_val.shape}")
print(f"The shape of the training data is X test: {X_test.shape} and y test: {y_test.shape}")

Splitting data: using val in training data for final training
The shape of the training data is X train: (90941, 7) and y train: (90941,)
The shape of the training data is X val: (29799, 7) and y val: (29799,)
The shape of the training data is X test: (48603, 7) and y test: (48603,)


## 03. TF-IDF Grid Search: Optimizing N-gram Range and Vocabulary Size
### 3.1 Simple Grid Search

In [ ]:
print("Start training")
gs_results = []
for (x, y) in [(1,2), (1,3), (1,5)]:
    for max_features in [1000, 10000, 100000]:
        vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(x, y),
            stop_words='english'
        )

        model = LogisticRegression(max_iter=10000)

        X_train_ngram = vectorizer.fit_transform(X_train['text'])
        X_val_ngram = vectorizer.transform(X_val['text'])

        model.fit(X_train_ngram, y_train)

        y_pred = model.predict(X_val_ngram)
        gs_results.append({
            'ngram_range': vectorizer.ngram_range,
            'max_features': vectorizer.max_features,
            'accuracy': accuracy_score(y_val, y_pred),
            'macro_f1': f1_score(y_val, y_pred, average='macro'),
        })

Start training


In [ ]:
df_results = pd.DataFrame(gs_results)
best_params = df_results.sort_values(by="accuracy", ascending=False).iloc[0]
print(f"Best parameters by accuracy: {best_params}")
print(gs_results)

### 3.2 Training & Evaluation Using Best Parameters

In [ ]:
print("Splitting data: using val in training data for final training")
X_train = data[data['split'] != 'test']
y_train = data[data['split'] != 'test'].label

X_test = data[data['split'] == 'test']
y_test = data[data['split'] == 'test'].label

print(f"The shape of the training data is X train: {X_train.shape} and y train: {y_train.shape}")
print(f"The shape of the training data is X test: {X_test.shape} and y test: {y_test.shape}")

In [ ]:
import ast

print(f"Start training: using {best_params}")

vectorizer = TfidfVectorizer(
    max_features=best_params.max_features,
    ngram_range= ast.literal_eval(best_params.ngram_range),
    stop_words='english'
)

model = LogisticRegression(max_iter=10000)

X_train_ngram = vectorizer.fit_transform(X_train['text'])
X_test_ngram = vectorizer.transform(X_test['text'])

model.fit(X_train_ngram, y_train)
y_pred = model.predict(X_test_ngram)

In [ ]:
results = pd.DataFrame({
    'model': 'LogisticRegression',
    'ngram_range': str(best_params.ngram_range),
    'max_iter': 10000,
    'max_features': best_params.max_features,
    'accuracy': accuracy_score(y_test, y_pred),
    'macro_f1': f1_score(y_test, y_pred, average='macro')
}, index=[0])
print(f"Final Results")
results

## 04. Analysis

Before analyzing the results two i